In [1]:
"""
Titanic Survival Prediction - Decision Tree Classifier
--------------------------------------------------------
Goal: predict whether a passenger survived (1) or not (0)
based on features like class, sex, age, fare, etc.

Steps:
1. Load the data
2. Clean it (handle missing values, convert text to numbers)
3. Pick features (inputs) and label (output = Survived)
4. Split into train/test sets
5. Train a Decision Tree
6. Score the model (accuracy)
"""

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Load the data
# ---------------------------------------------------------
df = pd.read_csv("train.csv")

print("Shape of dataset:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())

# ---------------------------------------------------------
# 2. Clean the data
# ---------------------------------------------------------

# Age has missing values -> fill with the median age
# (median is safer than mean here since Age has some outliers)
df["Age"] = df["Age"].fillna(df["Age"].median())

# Embarked has a couple missing values -> fill with the most common port
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin is missing for ~77% of passengers -> too sparse to be useful, drop it
df = df.drop(columns=["Cabin"])

# Name and Ticket are free text / IDs, not useful for a simple tree -> drop them
df = df.drop(columns=["Name", "Ticket", "PassengerId"])

# Convert text categories into numbers, since sklearn only understands numbers
# Sex: male/female -> 0/1
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

# Embarked: S/C/Q -> 0/1/2 (using one-hot encoding is more "correct",
# but label mapping keeps things simple and readable for a first model)
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

print("\nCleaned data preview:")
print(df.head())

# ---------------------------------------------------------
# 3. Features (X) and label (y)
# ---------------------------------------------------------
X = df.drop(columns=["Survived"])   # everything except the answer
y = df["Survived"]                  # the answer we want to predict

# ---------------------------------------------------------
# 4. Train/test split
#    We train on 80% of the data and test on the remaining 20%
#    that the model has NEVER seen, to check it actually learned
#    something (not just memorized).
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining rows: {len(X_train)}, Test rows: {len(X_test)}")

# ---------------------------------------------------------
# 5. Train the Decision Tree
#    max_depth limits how many questions deep the tree can go.
#    Without a limit, a decision tree will keep splitting until
#    every leaf is pure -> memorizes the training data -> overfits
#    -> does great on training data but poorly on new data.
# ---------------------------------------------------------
model = DecisionTreeClassifier(
    criterion="entropy",   # use information gain (same math from before) to pick splits
    max_depth=5,            # cap tree depth to avoid overfitting
    random_state=42
)
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. Evaluate the model
# ---------------------------------------------------------
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n=== Model Score ===")
print(f"Accuracy on test set: {accuracy:.3f}  ({accuracy*100:.1f}%)")

# Cross-validation gives a more reliable score by testing on
# 5 different train/test splits and averaging the results
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"5-fold Cross-Validation Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print("\nConfusion Matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

# ---------------------------------------------------------
# 7. Which features mattered most?
# ---------------------------------------------------------
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance (higher = more useful for splitting):")
print(importances)

# ---------------------------------------------------------
# 8. Print the tree itself (first few levels) so you can
#    see the actual questions it's asking
# ---------------------------------------------------------
print("\n=== Tree structure ===")
print(export_text(model, feature_names=list(X.columns), max_depth=3))

Shape of dataset: (891, 12)

Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Cleaned data preview:
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0         0       3    0  22.0      1      0   7.2500         0
1         1       1    1  38.0      1      0  71.2833         1
2         1       3    1  26.0      0      0   7.9250         0
3         1       1    1  35.0      1      0  53.1000         0
4         0       3    0  35.0      0      0   8.0500         0

Training rows: 712, Test rows: 179

=== Model Score ===
Accuracy on test set: 0.804  (80.4%)
5-fold Cross-Validation Accuracy: 0.813 (+/- 0.024)

Confusion Matrix (rows = actual, columns = predicted):
[[95 10]
 [25 49]]

Classification Report:
                 precision    recall  f1-score   su